# 21.9 — SMT solvers

SMT combines Boolean search with theory reasoning, so assignments must satisfy both clauses and arithmetic meaning.

SMT extends SAT from pure Booleans to formulas whose atoms have arithmetic meaning. The Boolean engine proposes candidate atom sets; the theory engine disposes by checking numeric feasibility.

Save a copy to Drive to edit.

In [ ]:

import itertools
import random

import matplotlib.pyplot as plt

random.seed(219)


def boolean_rows(atoms):
    for values in itertools.product([False, True], repeat=len(atoms)):
        yield dict(zip(atoms, values))


def eval_boolean(formula, assignment):
    kind = formula[0]
    if kind == "atom":
        return assignment[formula[1]]
    if kind == "and":
        return all(eval_boolean(item, assignment) for item in formula[1])
    if kind == "or":
        return any(eval_boolean(item, assignment) for item in formula[1])
    if kind == "not":
        return not eval_boolean(formula[1], assignment)
    if kind == "implies":
        return (not eval_boolean(formula[1], assignment)) or eval_boolean(formula[2], assignment)
    raise ValueError(kind)


def interval_feasible(chosen_atoms, interval_atoms):
    lower = -float("inf")
    upper = float("inf")
    reasons = []
    for atom in chosen_atoms:
        variable, operator, value = interval_atoms[atom]
        if operator == ">=":
            lower = max(lower, value)
            reasons.append(atom + " raises lower to " + str(lower))
        if operator == "<=":
            upper = min(upper, value)
            reasons.append(atom + " lowers upper to " + str(upper))
    feasible = lower <= upper
    interval = (lower, upper)
    return feasible, interval, reasons


def build_smt_ladder():
    rungs = []
    rungs.append({
        "name": "D1 two-atom interval SMT",
        "atoms": ["P", "Q"],
        "formula": ("and", [("atom", "P"), ("atom", "Q")]),
        "interval_atoms": {"P": ("x", ">=", 2), "Q": ("x", "<=", 5)},
        "expected_sat": True,
    })
    rungs.append({
        "name": "D2 several nonempty intervals",
        "atoms": ["A", "B", "C", "D"],
        "formula": ("and", [("atom", "A"), ("atom", "B"), ("implies", ("atom", "C"), ("atom", "D"))]),
        "interval_atoms": {"A": ("x", ">=", 1), "B": ("x", "<=", 8), "C": ("x", ">=", 3), "D": ("x", "<=", 6)},
        "expected_sat": True,
    })
    rungs.append({
        "name": "D3 conflicting branch",
        "atoms": ["A", "B", "C"],
        "formula": ("and", [("atom", "A"), ("atom", "B"), ("atom", "C")]),
        "interval_atoms": {"A": ("x", ">=", 4), "B": ("x", "<=", 1), "C": ("x", ">=", 0)},
        "expected_sat": False,
    })
    rungs.append({
        "name": "D4 program path constraints",
        "atoms": ["enter", "after_check", "fast", "safe", "exit"],
        "formula": ("and", [("atom", "enter"), ("implies", ("atom", "enter"), ("atom", "after_check")), ("implies", ("atom", "fast"), ("atom", "safe")), ("atom", "exit")]),
        "interval_atoms": {"enter": ("x", ">=", 0), "after_check": ("x", ">=", 2), "fast": ("x", "<=", 3), "safe": ("x", "<=", 6), "exit": ("x", "<=", 9)},
        "expected_sat": True,
    })
    rungs.append({
        "name": "D5 configuration with arithmetic infeasibility",
        "atoms": ["base", "gpu", "audit", "pro", "budget", "memory", "latency", "region"],
        "formula": ("and", [("atom", "base"), ("implies", ("atom", "gpu"), ("atom", "memory")), ("implies", ("atom", "pro"), ("atom", "audit")), ("atom", "gpu"), ("atom", "pro"), ("atom", "budget"), ("atom", "latency")]),
        "interval_atoms": {"base": ("x", ">=", 0), "gpu": ("x", ">=", 7), "audit": ("x", "<=", 9), "pro": ("x", ">=", 4), "budget": ("x", "<=", 6), "memory": ("x", ">=", 5), "latency": ("x", "<=", 8), "region": ("x", ">=", 1)},
        "expected_sat": False,
    })
    return rungs


## The concept, built once (D1)

SMT abstracts theory atoms into Boolean variables, then asks whether the selected atoms can hold together. The lesson's Boolean formula $P\land Q$ has $1$ satisfying row out of $4$, and $x\ge2$ with $x\le5$ gives the feasible interval $[2,5]$.

In [ ]:

lesson_atoms = ["P", "Q"]
lesson_formula = ("and", [("atom", "P"), ("atom", "Q")])
lesson_interval_atoms = {"P": ("x", ">=", 2), "Q": ("x", "<=", 5)}
lesson_boolean_rows = list(boolean_rows(lesson_atoms))
lesson_boolean_hits = [row for row in lesson_boolean_rows if eval_boolean(lesson_formula, row)]
lesson_feasible, lesson_interval, lesson_reasons = interval_feasible(["P", "Q"], lesson_interval_atoms)

assert len(lesson_boolean_rows) == 4
assert len(lesson_boolean_hits) == 1
assert lesson_interval == (2, 5)
assert lesson_feasible is True

print("Boolean rows", len(lesson_boolean_rows))
print("Boolean rows satisfying P and Q", len(lesson_boolean_hits))
print("theory interval", lesson_interval)


The mini solver enumerates Boolean candidates, keeps those satisfying the Boolean skeleton, and rejects candidates whose arithmetic interval is empty. This catches the lesson contradiction $x\ge4$ and $x\le1$ because $4\gt1$.

In [ ]:

def mini_smt(atoms, boolean_formula, interval_atoms):
    accepted = []
    rejected = []
    candidates = 0
    for assignment in boolean_rows(atoms):
        if not eval_boolean(boolean_formula, assignment):
            continue
        candidates += 1
        chosen = [atom for atom in atoms if assignment[atom]]
        feasible, interval, reasons = interval_feasible(chosen, interval_atoms)
        record = {
            "assignment": assignment,
            "chosen": chosen,
            "interval": interval,
            "reasons": reasons,
        }
        if feasible:
            accepted.append(record)
        else:
            rejected.append(record)
    return {
        "sat": len(accepted) > 0,
        "accepted": accepted,
        "rejected": rejected,
        "candidates": candidates,
    }

conflict_atoms = {"R": ("x", ">=", 4), "S": ("x", "<=", 1)}
conflict_feasible, conflict_interval, conflict_reasons = interval_feasible(["R", "S"], conflict_atoms)

assert conflict_feasible is False
assert conflict_interval == (4, 1)

lesson_result = mini_smt(lesson_atoms, lesson_formula, lesson_interval_atoms)
assert lesson_result["sat"] is True
print(lesson_result)


## The dataset ladder

D1–D5 add more Boolean structure and theory atoms, ending with a D5 branch that looks Boolean-valid but is arithmetically impossible.

In [ ]:

smt_rungs = build_smt_ladder()

for rung in smt_rungs:
    print(rung["name"])
    print("  atoms", len(rung["atoms"]))
    print("  interval atoms", list(rung["interval_atoms"].items())[:3])


## Run the same method across D1–D5

In [ ]:

smt_results = []

for rung in smt_rungs:
    result = mini_smt(rung["atoms"], rung["formula"], rung["interval_atoms"])
    assert result["sat"] == rung["expected_sat"]
    smt_results.append({
        "name": rung["name"],
        "atoms": len(rung["atoms"]),
        "sat": result["sat"],
        "candidates": result["candidates"],
        "accepted": len(result["accepted"]),
        "rejected": len(result["rejected"]),
        "result": result,
    })

print("rung | sat | atoms | bool candidates | accepted | theory rejected")
for item in smt_results:
    print(item["name"], item["sat"], item["atoms"], item["candidates"], item["accepted"], item["rejected"])


## Results visualization

In [ ]:

fig, axes = plt.subplots(2, 5, figsize=(18, 7))

for index, item in enumerate(smt_results):
    ax = axes[0][index]
    ax.set_title("D" + str(index + 1))
    ax.axis("off")
    accepted = item["result"]["accepted"][:2]
    rejected = item["result"]["rejected"][:2]
    lines = ["accepted " + str(len(item["result"]["accepted"])), "rejected " + str(len(item["result"]["rejected"]))]
    for record in accepted + rejected:
        lines.append(str(record["chosen"]) + " -> " + str(record["interval"]))
    ax.text(0.02, 0.95, "\n".join(lines), va="top", family="monospace", fontsize=8)

x_values = list(range(1, 6))
accepted_values = [item["accepted"] for item in smt_results]
rejected_values = [item["rejected"] for item in smt_results]
axes[1][0].plot(x_values, accepted_values, marker="o", label="accepted")
axes[1][0].plot(x_values, rejected_values, marker="s", label="theory rejected")
axes[1][0].set_xticks(x_values)
axes[1][0].set_xlabel("rung")
axes[1][0].set_ylabel("candidates")
axes[1][0].set_title("accepted vs rejected candidates")
axes[1][0].legend()
for ax in axes[1][1:]:
    ax.axis("off")
plt.tight_layout()


## Pitfall on the hardest rung

Pitfall: confusing a surviving Boolean candidate with a proved SMT answer. D5 has a Boolean branch, but the theory interval is empty.

In [ ]:

d5 = smt_rungs[-1]
boolean_only = []
for assignment in boolean_rows(d5["atoms"]):
    if eval_boolean(d5["formula"], assignment):
        boolean_only.append(assignment)
fixed_result = mini_smt(d5["atoms"], d5["formula"], d5["interval_atoms"])

print("Boolean-only survivors", len(boolean_only))
print("SMT accepted", len(fixed_result["accepted"]))
print("SMT rejected", len(fixed_result["rejected"]))
if fixed_result["rejected"]:
    print("first rejection interval", fixed_result["rejected"][0]["interval"])
    print("first rejection reasons", fixed_result["rejected"][0]["reasons"])


## Evaluate it + Practice

- Metric: satisfiability, Boolean candidates, theory rejections.
- No-skill baseline: accept every Boolean satisfying row.
- Cheap sanity check: D1 has 4 Boolean rows, 1 Boolean hit, and interval [2, 5].
- Ablation: skip interval_feasible and D5 becomes falsely accepted.
- Failure signals: a branch is reported without its arithmetic interval.

Practice 1: Change the D2 instance by one constraint and predict the metric before running.

Practice 2: Add one contradictory or noisy condition to D4 and explain how the trace changes.

Practice 3: On D5, record the first step where pruning or explanation prevents a wrong answer.